# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd

DATA_CANDIDATES = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
local_path = next((path for path in DATA_CANDIDATES if path.exists()), None)
if local_path is not None:
    df = pd.read_csv(local_path)
    data_source = str(local_path)
else:
    data_url = "https://raw.githubusercontent.com/somnathsutra/ML-01-ASSIGNMENT/main/data/raw/content_refresh_anonymized.csv"
    with urlopen(data_url) as response:
        df = pd.read_csv(BytesIO(response.read()))
    data_source = data_url

# The label is computed from the final 30-day trend. Keep only prior-window and
# stable content descriptors in the honest feature vector.
label = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)

safe_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
]
safe_categorical = [
    "competition_level", "content_type", "main_intent", "provider_used",
    "model_used", "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier",
]
safe_columns = safe_numeric + safe_categorical

assert set(safe_columns).issubset(df.columns)
assert len(df) == 30_000
assert label.nunique() == 2

feature_frame = df[safe_columns].copy()
for column in safe_numeric:
    feature_frame[f"has_{column}"] = feature_frame[column].notna().astype(int)
    feature_frame[column] = pd.to_numeric(feature_frame[column], errors="coerce")
    feature_frame[column] = feature_frame[column].fillna(feature_frame[column].median())
for column in safe_categorical:
    feature_frame[column] = feature_frame[column].astype("string").fillna("unknown")
feature_frame = pd.get_dummies(feature_frame, columns=safe_categorical, dtype=int)

print(f"Loaded {len(df):,} rows from {data_source}")
print(f"Safe raw fields: {len(safe_columns)}")
print(f"Encoded feature columns: {feature_frame.shape[1]}")
print(f"Positive label rate: {label.mean():.3f}")
print("Grouped split key retained separately: client_id")

Loaded 30,000 rows from https://raw.githubusercontent.com/somnathsutra/ML-01-ASSIGNMENT/main/data/raw/content_refresh_anonymized.csv
Safe raw fields: 20
Encoded feature columns: 60
Positive label rate: 0.542
Grouped split key retained separately: client_id


## 2. Feature notes (meaning, missing, categorical, available-when?)

The feature vector uses stable content metadata and the **previous** 30-day performance window. Numeric missingness is represented by a `has_` indicator and then median-filled; categorical blanks become `unknown`. The final 30-day fields, overlapping 90-day aggregates, trend fields, and identifiers are deliberately absent because they are unavailable or contaminated at the prediction moment. `client_id` is retained outside the matrix for grouped validation only.

In [2]:
feature_notes = pd.DataFrame([
    {
        "family": "static numeric",
        "fields": ", ".join(["search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days", "age_tier_order", "days_since_last_update"]),
        "availability": "before review",
        "missing_handling": "has_ flag plus median fill",
    },
    {
        "family": "prior-window numeric",
        "fields": ", ".join(["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]),
        "availability": "before final-30d label window",
        "missing_handling": "has_ flag plus median fill",
    },
    {
        "family": "categorical metadata",
        "fields": ", ".join(safe_categorical),
        "availability": "before review",
        "missing_handling": "unknown category",
    },
])
print(feature_notes.to_string(index=False))
print(f"Any missing values left in encoded matrix: {int(feature_frame.isna().sum().sum())}")
assert feature_frame.isna().sum().sum() == 0

              family                                                                                                                              fields                  availability           missing_handling
      static numeric                   search_volume, competition, cpc, word_count, char_count, content_age_days, age_tier_order, days_since_last_update                 before review has_ flag plus median fill
prior-window numeric                                                                            impressions_prev_30d, clicks_prev_30d, sessions_prev_30d before final-30d label window has_ flag plus median fill
categorical metadata competition_level, content_type, main_intent, provider_used, model_used, age_tier, freshness_tier, word_count_tier, char_count_tier                 before review           unknown category
Any missing values left in encoded matrix: 0


## 3. The leakage hunt

Three attacks are covered below: label-derived fields (`trend_direction`, `trend_pct`), overlapping outcome windows (last-30-day and 90-day aggregates), and identifiers/product decisions. The final assertions ensure none of those fields enter the feature matrix.

In [3]:
label_derived = {"trend_direction", "trend_pct", "is_declining_label"}
overlapping_windows = {
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "impressions_last_30d",
    "clicks_last_30d", "sessions_last_30d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
}
identifiers_and_decisions = {
    "content_id", "client_id", "health_score", "priority_score",
    "action_type", "refresh_tier",
}

raw_feature_set = set(safe_columns)
encoded_source_columns = set(safe_columns)

print(f"Label-derived fields in raw vector: {sorted(raw_feature_set & label_derived)}")
print(f"Overlapping outcome-window fields in raw vector: {sorted(raw_feature_set & overlapping_windows)}")
print(f"Identifiers/product decisions in raw vector: {sorted(raw_feature_set & identifiers_and_decisions)}")
print(f"Calendar fields available: {[column for column in df.columns if column.lower() in {'date', 'month', 'report_date'}]}")

assert not raw_feature_set & label_derived
assert not raw_feature_set & overlapping_windows
assert not raw_feature_set & identifiers_and_decisions
assert "client_id" not in encoded_source_columns
assert "content_id" not in encoded_source_columns
print("Leakage checks passed: the encoded matrix contains no label, overlapping outcome, ID, or product-decision fields.")

Label-derived fields in raw vector: []
Overlapping outcome-window fields in raw vector: []
Identifiers/product decisions in raw vector: []
Calendar fields available: []
Leakage checks passed: the encoded matrix contains no label, overlapping outcome, ID, or product-decision fields.


## 4. What I excluded and why

- `trend_direction`, `trend_pct`: label source or sibling; including either reveals the outcome.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`: overlap the final-30-day trend label window.
- All `*_90d` performance totals, current rates, and `avg_position`: their windows overlap the label window, so they are not strictly pre-label.
- `content_id`, `client_id`: pseudonymous identifiers; retained only for grouping and client-aware validation.
- `health_score`, `priority_score`, `action_type`, `refresh_tier`: product decisions; they would reproduce an existing rule rather than test independent signal.
- Raw queries, URLs, client names, or private identifiers: not part of the public-safe feature contract.

In [4]:
excluded_fields = sorted(
    (label_derived | overlapping_windows | identifiers_and_decisions) & set(df.columns)
)
feature_columns = set(feature_frame.columns)

print("Excluded fields present in the source:")
print(excluded_fields)
print(f"Final encoded feature count: {len(feature_columns)}")
print("Validation design: keep client_id outside X and use a grouped train/test split in the modeling notebook.")

assert "client_id" not in feature_columns
assert "content_id" not in feature_columns
assert not any(field in feature_columns for field in label_derived)
assert not any(field in feature_columns for field in overlapping_windows)
print("Final exclusion audit passed.")

Excluded fields present in the source:
['ai_sessions_90d', 'ai_traffic_pct', 'avg_position', 'clicks_90d', 'clicks_last_30d', 'client_id', 'content_id', 'ctr', 'days_with_impressions', 'days_with_sessions', 'engaged_sessions_90d', 'engagement_rate', 'impressions_90d', 'impressions_last_30d', 'pageviews_90d', 'scroll_events_90d', 'scroll_rate', 'sessions_90d', 'sessions_last_30d', 'trend_direction', 'trend_pct', 'users_90d']
Final encoded feature count: 60
Validation design: keep client_id outside X and use a grouped train/test split in the modeling notebook.
Final exclusion audit passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.